# I want to demonstrate the idea of tool calling

In [3]:
# make sure you have the openai api key set in your environment variable
# I will push this to the repo later
from openai import OpenAI

In [4]:
client = OpenAI()

First just an LLM input and output.

In [7]:
input = "Top 5 things an instructor should never do during a live code demonstration."

response = client.responses.create(
    model="gpt-4.1-mini",
    input=input,
)


response.output_text

'Here are the top 5 things an instructor should never do during a live code demonstration:\n\n1. **Read code verbatim without explanation**  \n   Simply reading through the code without explaining the reasoning, logic, or context can lose the audience’s engagement and understanding.\n\n2. **Ignore compilation or runtime errors**  \n   Skipping over errors or not addressing them clearly can confuse learners and miss valuable teaching moments on debugging and problem-solving.\n\n3. **Move too quickly or too slowly**  \n   Coding too fast can overwhelm learners, while going too slow can bore them. Finding a balanced pace is crucial.\n\n4. **Skip fundamental concepts or assumptions**  \n   Assuming everyone knows certain concepts without a brief recap can leave beginners behind. It’s important to clarify assumptions.\n\n5. **Read code from a script or slides without interaction**  \n   Not engaging with the learners through questions, pauses, or live problem-solving reduces interactivity a

Let's integrate a tool into this model!

In [ ]:
path_to_file = "../README.md"


def read_file_tool(path_to_file: str):
    with open(path_to_file, "r") as file:
        content = file.read()
    
    return content

read_file_tool(path_to_file)

'# O\'Reilly Live Training: Building AI Agents with OpenAI\'s Agents SDK\n\nA comprehensive educational repository for learning to build sophisticated AI agents using OpenAI\'s Agents SDK. This course progresses from basic agent creation to advanced multi-agent workflows, tool integration, and voice capabilities.\n\n## 🚀 Quick Start\n\n### Prerequisites\n- Python 3.11+\n- Conda package manager\n- OpenAI API key\n\n### Setup\n```bash\n# Complete setup from scratch\nmake all\n\n# Or step by step:\nmake conda-create          # Create conda environment\nmake env-setup            # Setup pip-tools and ipykernel  \nmake repo-setup           # Initialize requirements\nmake notebook-setup       # Install Jupyter kernel\nmake env-update           # Update dependencies\n```\n\n### Manual Setup\n```bash\nconda activate openai-agents\npip install openai-agents\n```\n\n## 📚 Course Structure\n\n### Learning Path (Sequential Notebooks)\n\nThe course is organized as a progressive series of Jupyter not

# Old School Tool Calling

In [9]:
def llm_with_file_tool(input: str):
    file_tool_prompt = f"""
    You are a helpful assistant that can read files.
    If you need to read a file, your output should be:
    read_file_tool([path_to_file])
    Here is the user's question:
    {input}
    """
    response = client.responses.create(
        model="gpt-4.1-mini",
        input=file_tool_prompt,
    )
    
    return response.output_text



input = "who won the NBA in 2025?"
llm_with_file_tool(input)

'The NBA champion for 2025 is not available as it is in the future relative to my knowledge cutoff date in June 2024. If you have any other questions or need information about previous NBA seasons, feel free to ask!'

In [10]:
input_that_requires_file_tool = "Summarize the README.md file located at: ../README.md"
llm_with_file_tool(input_that_requires_file_tool)

'read_file_tool(["../README.md"])'

# Tool Calling with Tool Schemas (Modern)

In [11]:
file_tool_schema = {
        "type": "function",
        "name": "read_file_tool",
        "description": "Read and return the content of a specified file.",
        "parameters": {
            "type": "object",
            "properties": {
                "path_to_file": {
                    "type": "string",
                    "description": "The file path to read, such as '../README.md'.",
                },
            },
            "required": ["path_to_file"],
        },
    }

def llm_with_file_tool_modern(input: str):
    response = client.responses.create(
        model="gpt-4.1-mini",
        input=input,
        tools=[file_tool_schema],
    )
    
    return response

input_that_requires_file_tool = "Summarize the ../README.md file."
output = llm_with_file_tool_modern(input_that_requires_file_tool)
output

Response(id='resp_00ada9f0f3f6893f0069cd1265d55081908546db7dd8358bd6', created_at=1775047269.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4.1-mini-2025-04-14', object='response', output=[ResponseFunctionToolCall(arguments='{"path_to_file":"../README.md"}', call_id='call_mbUFykcKmeHnAo9M9SxHtJxW', name='read_file_tool', type='function_call', id='fc_00ada9f0f3f6893f0069cd1266b03481909929b792383394cd', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='read_file_tool', parameters={'type': 'object', 'properties': {'path_to_file': {'type': 'string', 'description': "The file path to read, such as '../README.md'."}}, 'required': ['path_to_file'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Read and return the content of a specified file.')], top_p=1.0, background=False, completed_at=1775047270.0, conversation=None, max_output_toke

In [26]:
output.output[0].arguments

'{"path_to_file":"../README.md"}'

In [35]:
import json

def execute_tool(output):
    tool_name = output.output[0].name
    if tool_name == "read_file_tool":
        arguments = output.output[0].arguments
        arguments_dict = json.loads(arguments)
        file_path = arguments_dict['path_to_file']
        
        return read_file_tool(arguments_dict['path_to_file'])

execute_tool(output)

'# O\'Reilly Live Training: Building AI Agents with OpenAI\'s Agents SDK\n\nA comprehensive educational repository for learning to build sophisticated AI agents using OpenAI\'s Agents SDK. This course progresses from basic agent creation to advanced multi-agent workflows, tool integration, and voice capabilities.\n\n## 🚀 Quick Start\n\n### Prerequisites\n- Python 3.11+\n- Conda package manager\n- OpenAI API key\n\n### Setup\n```bash\n# Complete setup from scratch\nmake all\n\n# Or step by step:\nmake conda-create          # Create conda environment\nmake env-setup            # Setup pip-tools and ipykernel  \nmake repo-setup           # Initialize requirements\nmake notebook-setup       # Install Jupyter kernel\nmake env-update           # Update dependencies\n```\n\n### Manual Setup\n```bash\nconda activate openai-agents\npip install openai-agents\n```\n\n## 📚 Course Structure\n\n### Learning Path (Sequential Notebooks)\n\nThe course is organized as a progressive series of Jupyter not

In [38]:
# action would be

input_that_requires_file_tool = "Summarize the ../README.md file in 2 sentences."
output = llm_with_file_tool_modern(input_that_requires_file_tool)
tool_output = execute_tool(output)
prompt_after_tool_call = f"""
Here is the user's question:
{input_that_requires_file_tool}
Here is the tool output:
{tool_output}
Here is the response from you:
{output.output_text}
Answer the user:
"""
output_after_tool_call = llm_with_file_tool_modern(prompt_after_tool_call)
print("FINAL ANSWER:")
print(output_after_tool_call.output_text)

FINAL ANSWER:
The README.md describes a comprehensive course and repository for building advanced AI agents using OpenAI's Agents SDK, featuring progressive Jupyter notebooks, core modules, tool integrations, and voice capabilities. It provides setup instructions, development commands, example applications, and emphasizes modular, real-world integration with detailed testing and code quality practices.
